# 🤖 Task 2 — FAQ Chatbot
### CodeAlpha Artificial Intelligence Internship

**Submitted by:** Navya
**Stack:** Python · NLTK · Scikit-learn (TF-IDF) · Cosine Similarity · Streamlit

---

## 📋 What this notebook does

| Step | Description |
|------|--------------|
| 1 | Install required Python packages |
| 2 | Download NLTK language resources |
| 3 | Build the FAQ knowledge base |
| 4 | Build the NLP preprocessing + TF-IDF + cosine-similarity matching engine |
| 5 | Test the chatbot logic directly in the notebook |
| 6 | Write the full Streamlit chat application to disk |
| 7 | Configure `ngrok` and launch a live, shareable chatbot |
| 8 | Stop the app when done |

> 💡 **How it works:** User questions are cleaned (lowercased, stopwords removed,
> lemmatized) and converted into TF-IDF vectors. We compare this vector against
> the TF-IDF vectors of all known FAQ questions using **cosine similarity**, and
> return the answer for the closest match — if the similarity score clears a
> confidence threshold.

## 📦 Step 1 — Install Dependencies

- **`nltk`** — tokenization, stopword removal, lemmatization
- **`scikit-learn`** — TF-IDF vectorizer + cosine similarity
- **`streamlit`** — the chat web app UI
- **`pyngrok`** — public URL tunnel

In [ ]:
!pip install nltk scikit-learn streamlit pyngrok -q

print("✅ All packages installed successfully!")

## 📚 Step 2 — Download NLTK Resources

We need a few NLTK corpora/models:
- **`punkt`** / **`punkt_tab`** — sentence & word tokenizer models
- **`stopwords`** — common words ("the", "is", "a"...) to filter out
- **`wordnet`** — used by the lemmatizer to reduce words to their root form

In [ ]:
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

print("✅ NLTK resources downloaded!")

## 🗂️ Step 3 — Build the FAQ Knowledge Base

This is our "database" of question–answer pairs covering core AI/ML topics.
Feel free to add more entries — the matching engine will automatically pick
them up.

In [ ]:
%%writefile faq_data.py
faqs = [
    {
        "question": "What is artificial intelligence?",
        "answer": "Artificial Intelligence (AI) is the simulation of human intelligence in machines programmed to think, learn, and problem-solve like humans. It includes subfields like machine learning, deep learning, and natural language processing."
    },
    {
        "question": "What is machine learning?",
        "answer": "Machine Learning (ML) is a subset of AI where algorithms learn patterns from data without being explicitly programmed. It improves automatically through experience using methods like supervised, unsupervised, and reinforcement learning."
    },
    {
        "question": "What is deep learning?",
        "answer": "Deep Learning is a subset of ML using neural networks with many layers (deep neural networks). It excels at tasks like image recognition, speech processing, and natural language understanding."
    },
    {
        "question": "What is natural language processing?",
        "answer": "Natural Language Processing (NLP) is the branch of AI focused on enabling computers to understand, interpret, and generate human language. Applications include chatbots, translation, sentiment analysis, and text summarization."
    },
    {
        "question": "What is a neural network?",
        "answer": "A neural network is a computational model inspired by the human brain. It consists of layers of interconnected nodes (neurons) that process data and learn to recognize patterns by adjusting connection weights during training."
    },
    {
        "question": "What is supervised learning?",
        "answer": "Supervised learning is a type of ML where the model is trained on labeled data — input-output pairs — so it learns to map inputs to correct outputs. Examples include classification and regression tasks."
    },
    {
        "question": "What is unsupervised learning?",
        "answer": "Unsupervised learning trains on unlabeled data to find hidden patterns or structure. Common techniques include clustering (e.g., K-Means) and dimensionality reduction (e.g., PCA)."
    },
    {
        "question": "What is reinforcement learning?",
        "answer": "Reinforcement Learning (RL) is an ML paradigm where an agent learns by interacting with an environment, receiving rewards or penalties. It powers applications like game-playing AIs (AlphaGo) and robotics."
    },
    {
        "question": "What is overfitting in machine learning?",
        "answer": "Overfitting occurs when a model learns the training data too well — including noise — and performs poorly on new, unseen data. It's prevented using techniques like regularization, dropout, and cross-validation."
    },
    {
        "question": "What is a large language model?",
        "answer": "A Large Language Model (LLM) is a deep learning model trained on massive text datasets to understand and generate human-like text. Examples include GPT-4, Claude, and LLaMA."
    },
    {
        "question": "What is transfer learning?",
        "answer": "Transfer learning involves taking a pre-trained model (e.g., BERT, ResNet) and fine-tuning it for a new, related task. It saves training time and works well when labeled data is limited."
    },
    {
        "question": "What is a convolutional neural network?",
        "answer": "A Convolutional Neural Network (CNN) is a deep learning architecture designed for processing structured grid data like images. It uses convolutional layers to detect features like edges, textures, and objects."
    },
    {
        "question": "What is the difference between AI and ML?",
        "answer": "AI is the broad concept of creating intelligent machines, while ML is a specific approach to achieve AI using data-driven learning. All ML is AI, but not all AI is ML — rule-based systems are AI without ML."
    },
    {
        "question": "What is Python used for in AI?",
        "answer": "Python is the dominant language in AI/ML due to its simplicity and rich ecosystem: NumPy, Pandas, Scikit-learn, TensorFlow, PyTorch, and Keras. It's used for data preprocessing, model building, and deployment."
    },
    {
        "question": "What is a random forest?",
        "answer": "Random Forest is an ensemble learning method that builds multiple decision trees and merges their predictions for better accuracy and robustness. It's widely used for classification and regression tasks."
    },
    {
        "question": "What is a support vector machine?",
        "answer": "Support Vector Machine (SVM) is a supervised ML algorithm that finds the optimal hyperplane to separate classes in feature space. It works well for high-dimensional data and binary classification."
    },
    {
        "question": "What is feature engineering?",
        "answer": "Feature engineering is the process of selecting, transforming, or creating input variables (features) from raw data to improve model performance. It requires domain knowledge and creativity."
    },
    {
        "question": "What is a generative adversarial network?",
        "answer": "A GAN consists of two neural networks — a generator and a discriminator — competing against each other. The generator creates fake data; the discriminator tries to detect it. Used in image synthesis, deepfakes, and data augmentation."
    },
    {
        "question": "What is computer vision?",
        "answer": "Computer Vision enables machines to interpret and understand visual information from images or videos. Applications include facial recognition, medical imaging, self-driving cars, and object detection."
    },
    {
        "question": "What is the bias-variance tradeoff?",
        "answer": "Bias refers to errors from overly simple models; variance refers to errors from overly complex models. The tradeoff is finding the right complexity — low bias and low variance — to achieve good generalization."
    },
    {
        "question": "What is data preprocessing?",
        "answer": "Data preprocessing involves cleaning and transforming raw data before feeding it into an ML model. Steps include handling missing values, encoding categorical variables, normalization, and splitting into train/test sets."
    },
    {
        "question": "What is a transformer model?",
        "answer": "Transformers are deep learning architectures using self-attention mechanisms to process sequential data in parallel. Introduced in 'Attention is All You Need' (2017), they power most modern NLP models like BERT and GPT."
    },
    {
        "question": "What is gradient descent?",
        "answer": "Gradient descent is an optimization algorithm used to minimize the loss function in ML models by iteratively adjusting parameters in the direction of the steepest descent (negative gradient)."
    },
    {
        "question": "What are hyperparameters?",
        "answer": "Hyperparameters are configuration settings set before training (e.g., learning rate, batch size, number of layers). Unlike model parameters, they are not learned from data and must be tuned manually or via grid/random search."
    },
    {
        "question": "What is YOLO in object detection?",
        "answer": "YOLO (You Only Look Once) is a real-time object detection algorithm that processes the entire image in a single pass through a neural network, predicting bounding boxes and class probabilities simultaneously."
    },
]


In [ ]:
from faq_data import faqs

print(f"✅ Loaded {len(faqs)} FAQ entries.")
print("\nSample entries:")
for item in faqs[:3]:
    print(f"  Q: {item['question']}")
    print(f"  A: {item['answer'][:80]}...\n")

## 🧠 Step 4 — Build the NLP Matching Engine

**Pipeline:**
1. `preprocess()` — lowercase → remove punctuation → tokenize → remove
   stopwords → lemmatize
2. `TfidfVectorizer` — converts every FAQ question into a TF-IDF vector
   (using unigrams + bigrams for better phrase matching)
3. `get_answer()` — preprocesses the user's query, vectorizes it, computes
   cosine similarity against every FAQ question, and returns the best match
   (only if the similarity score clears the confidence threshold)

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))


def preprocess(text):
    """
    Clean and normalize text for matching:
    1. Lowercase everything
    2. Strip out anything that isn't a letter, digit, or whitespace
    3. Tokenize into words
    4. Remove stopwords
    5. Lemmatize each remaining word (e.g. "running" -> "run")
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return " ".join(tokens)


# Pre-process every FAQ question once, up front
faq_questions = [item["question"] for item in faqs]
processed_questions = [preprocess(q) for q in faq_questions]

# Build the TF-IDF matrix (unigrams + bigrams for short-phrase matching)
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(processed_questions)

print(f"✅ TF-IDF matrix built: {tfidf_matrix.shape[0]} questions x {tfidf_matrix.shape[1]} features")


def get_answer(user_query, threshold=0.15):
    """
    Given a raw user query string, return:
        (answer_text_or_None, similarity_score, matched_question_or_None)

    If the best similarity score is below `threshold`, we treat it as
    "no confident match" and return None for the answer.
    """
    processed_query = preprocess(user_query)
    query_vector = vectorizer.transform([processed_query])

    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    best_index = int(np.argmax(similarities))
    best_score = float(similarities[best_index])

    if best_score < threshold:
        return None, best_score, None

    return faqs[best_index]["answer"], best_score, faqs[best_index]["question"]

## 🧪 Step 5 — Test the Chatbot Logic

Let's sanity-check the matching engine with a handful of test queries,
including one that's intentionally out-of-scope (to confirm the
confidence threshold correctly rejects it).

In [ ]:
test_queries = [
    "explain machine learning",
    "what is deep learning?",
    "how does YOLO work?",
    "tell me about transformers",
    "what is pizza?",          # out-of-scope — should be rejected
]

print("🧪 Chatbot Engine Test")
print("=" * 70)

for query in test_queries:
    answer, score, matched_question = get_answer(query)

    print(f"❓ Query     : {query}")
    if matched_question:
        print(f"🎯 Matched Q : {matched_question}")
        print(f"📊 Confidence: {score:.3f}  ({score * 100:.0f}%)")
        print(f"✅ Answer    : {answer}")
    else:
        print(f"📊 Confidence: {score:.3f}  ({score * 100:.0f}%)  — below threshold")
        print("⚠️  No confident match found.")
    print("-" * 70)

## 📝 Step 6 — Write the Streamlit Chat Application

This writes the full chat UI to `app.py`. It reuses the exact same
preprocessing → TF-IDF → cosine-similarity logic from Step 4, wrapped in a
clean chat interface.

**Features:**
- Chat-bubble style UI (user messages right-aligned, bot left-aligned)
- Quick-suggestion buttons for first-time users
- Confidence score + matched question shown under every bot reply
- Low-confidence warning banner
- Clear-chat button

In [ ]:
%%writefile app.py
import streamlit as st
import nltk
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import time
from faq_data import faqs

# ─── Download NLTK data ────────────────────────────────────────────────────────
@st.cache_resource
def download_nltk():
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt_tab', quiet=True)

download_nltk()

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ─── Page Config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="AskAI — FAQ Chatbot",
    page_icon="🤖",
    layout="centered"
)

# ─── CSS ───────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Space+Grotesk:wght@500;700&display=swap');

html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
.stApp { background: #0d1117; min-height: 100vh; }
#MainMenu, footer, header { visibility: hidden; }
.block-container { padding-top: 1.5rem; padding-bottom: 2rem; max-width: 780px; }

/* Hero */
.hero {
    text-align: center;
    padding: 2rem 1rem 1rem;
    border-bottom: 1px solid rgba(255,255,255,0.06);
    margin-bottom: 1.5rem;
}
.hero h1 {
    font-family: 'Space Grotesk', sans-serif;
    font-size: 2.2rem;
    font-weight: 700;
    color: #fff;
    margin-bottom: 0.3rem;
}
.hero h1 .accent { color: #22d3ee; }
.hero p { color: #64748b; font-size: 0.9rem; }
.badge {
    display: inline-block;
    background: rgba(34, 211, 238, 0.1);
    color: #22d3ee;
    border: 1px solid rgba(34, 211, 238, 0.2);
    border-radius: 20px;
    padding: 0.2rem 0.8rem;
    font-size: 0.7rem;
    font-weight: 600;
    letter-spacing: 1px;
    margin-bottom: 0.8rem;
}

/* Chat Container */
.chat-container {
    display: flex;
    flex-direction: column;
    gap: 1rem;
    margin-bottom: 1rem;
}

/* User Message */
.msg-user {
    display: flex;
    justify-content: flex-end;
    align-items: flex-start;
    gap: 0.6rem;
}
.bubble-user {
    background: linear-gradient(135deg, #0ea5e9, #22d3ee);
    color: #fff;
    border-radius: 18px 18px 4px 18px;
    padding: 0.75rem 1.1rem;
    max-width: 75%;
    font-size: 0.92rem;
    line-height: 1.5;
    box-shadow: 0 2px 12px rgba(14,165,233,0.25);
}
.avatar-user {
    width: 32px; height: 32px;
    background: #0ea5e9;
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 0.85rem;
    flex-shrink: 0;
    margin-top: 2px;
}

/* Bot Message */
.msg-bot {
    display: flex;
    justify-content: flex-start;
    align-items: flex-start;
    gap: 0.6rem;
}
.bubble-bot {
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.08);
    color: #e2e8f0;
    border-radius: 18px 18px 18px 4px;
    padding: 0.75rem 1.1rem;
    max-width: 80%;
    font-size: 0.92rem;
    line-height: 1.6;
}
.avatar-bot {
    width: 32px; height: 32px;
    background: rgba(34,211,238,0.15);
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 0.9rem;
    flex-shrink: 0;
    margin-top: 2px;
}
.confidence-bar {
    margin-top: 0.5rem;
    padding-top: 0.5rem;
    border-top: 1px solid rgba(255,255,255,0.06);
    font-size: 0.72rem;
    color: #64748b;
}
.conf-fill {
    display: inline-block;
    height: 3px;
    border-radius: 2px;
    background: #22d3ee;
    vertical-align: middle;
    margin: 0 0.4rem;
}

/* Welcome message */
.welcome-box {
    background: rgba(34,211,238,0.05);
    border: 1px solid rgba(34,211,238,0.15);
    border-radius: 12px;
    padding: 1.2rem 1.4rem;
    color: #94a3b8;
    font-size: 0.88rem;
    line-height: 1.6;
    margin-bottom: 1rem;
}
.welcome-box strong { color: #22d3ee; }

/* Suggestion chips */
.chips { display: flex; flex-wrap: wrap; gap: 0.5rem; margin-bottom: 1.2rem; }
.chip {
    background: rgba(255,255,255,0.05);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 20px;
    padding: 0.3rem 0.85rem;
    color: #94a3b8;
    font-size: 0.77rem;
    cursor: pointer;
}

/* Low confidence */
.low-conf {
    background: rgba(251,146,60,0.08);
    border: 1px solid rgba(251,146,60,0.2);
    color: #fb923c;
    border-radius: 8px;
    padding: 0.5rem 0.8rem;
    font-size: 0.82rem;
    margin-top: 0.4rem;
}

/* Footer */
.footer { text-align:center; color: #334155; font-size:0.73rem; margin-top:2rem; }
</style>
""", unsafe_allow_html=True)

# ─── NLP Pipeline ──────────────────────────────────────────────────────────────
@st.cache_resource
def build_nlp_engine():
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    def preprocess(text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        tokens = nltk.word_tokenize(text)
        tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
        return ' '.join(tokens)

    questions = [f["question"] for f in faqs]
    processed = [preprocess(q) for q in questions]

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(processed)

    return vectorizer, tfidf_matrix, preprocess

vectorizer, tfidf_matrix, preprocess = build_nlp_engine()

def get_answer(user_query, threshold=0.15):
    processed_q = preprocess(user_query)
    query_vec = vectorizer.transform([processed_q])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    best_idx = int(np.argmax(similarities))
    best_score = float(similarities[best_idx])

    if best_score < threshold:
        return None, best_score, None

    return faqs[best_idx]["answer"], best_score, faqs[best_idx]["question"]

# ─── Init session ─────────────────────────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []
if "input_key" not in st.session_state:
    st.session_state.input_key = 0

# ─── Hero ─────────────────────────────────────────────────────────────────────
st.markdown("""
<div class="hero">
    <div class="badge">NLP POWERED · COSINE SIMILARITY</div>
    <h1>Ask<span class="accent">AI</span></h1>
    <p>Your intelligent FAQ assistant for Artificial Intelligence topics</p>
</div>
""", unsafe_allow_html=True)

# ─── Welcome / Suggestions ────────────────────────────────────────────────────
if not st.session_state.messages:
    st.markdown("""
    <div class="welcome-box">
        👋 Hi! I'm <strong>AskAI</strong>, your AI knowledge assistant.<br>
        I can answer questions on <strong>Machine Learning, Deep Learning, NLP, Neural Networks</strong> and more.<br>
        Try one of the suggestions below or type your own question!
    </div>
    """, unsafe_allow_html=True)

    suggestions = [
        "What is machine learning?",
        "Explain deep learning",
        "What is a neural network?",
        "What is overfitting?",
        "How does YOLO work?",
    ]
    cols = st.columns(len(suggestions))
    for i, sug in enumerate(suggestions):
        if cols[i].button(sug, key=f"sug_{i}", use_container_width=True):
            st.session_state.messages.append({"role": "user", "content": sug})
            answer, score, matched_q = get_answer(sug)
            if answer:
                st.session_state.messages.append({
                    "role": "bot", "content": answer,
                    "score": score, "matched": matched_q
                })
            else:
                st.session_state.messages.append({
                    "role": "bot",
                    "content": "I couldn't find a confident match. Try rephrasing your question.",
                    "score": score, "matched": None
                })
            st.rerun()

# ─── Chat History ─────────────────────────────────────────────────────────────
for msg in st.session_state.messages:
    if msg["role"] == "user":
        st.markdown(f"""
        <div class="msg-user">
            <div class="bubble-user">{msg['content']}</div>
            <div class="avatar-user">👤</div>
        </div>
        """, unsafe_allow_html=True)
    else:
        conf_pct = int(msg.get('score', 0) * 100)
        bar_width = min(conf_pct * 1.5, 100)
        matched_text = f"Matched: <em>{msg.get('matched', '')}</em>" if msg.get('matched') else ""
        low_conf_html = ""
        if msg.get('score', 1) < 0.3 and msg.get('matched'):
            low_conf_html = '<div class="low-conf">⚠️ Low confidence match — consider rephrasing</div>'

        st.markdown(f"""
        <div class="msg-bot">
            <div class="avatar-bot">🤖</div>
            <div class="bubble-bot">
                {msg['content']}
                {low_conf_html}
                <div class="confidence-bar">
                    Confidence <span class="conf-fill" style="width:{bar_width}px"></span> {conf_pct}%
                    &nbsp;·&nbsp; {matched_text}
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)

# ─── Input ────────────────────────────────────────────────────────────────────
st.markdown("<div style='height:0.8rem'></div>", unsafe_allow_html=True)
col_in, col_send = st.columns([5, 1])
with col_in:
    user_input = st.text_input(
        "",
        placeholder="Ask me anything about AI, ML, Deep Learning...",
        key=f"user_input_{st.session_state.input_key}",
        label_visibility="collapsed"
    )
with col_send:
    send = st.button("Send ➤", use_container_width=True, type="primary")

if send and user_input.strip():
    st.session_state.messages.append({"role": "user", "content": user_input.strip()})
    with st.spinner("Thinking..."):
        time.sleep(0.3)
        answer, score, matched_q = get_answer(user_input.strip())
    if answer:
        st.session_state.messages.append({
            "role": "bot", "content": answer,
            "score": score, "matched": matched_q
        })
    else:
        st.session_state.messages.append({
            "role": "bot",
            "content": "I'm not confident about that one. Please try rephrasing or ask something else about AI/ML.",
            "score": score, "matched": None
        })
    st.session_state.input_key += 1
    st.rerun()

# ─── Clear Chat ───────────────────────────────────────────────────────────────
if st.session_state.messages:
    if st.button("🗑️ Clear Chat", use_container_width=False):
        st.session_state.messages = []
        st.session_state.input_key += 1
        st.rerun()

# ─── Footer ───────────────────────────────────────────────────────────────────
st.markdown("""
<div class="footer">
    Built with ❤️ by Navya · CodeAlpha AI Internship · Task 2 — FAQ Chatbot · NLP + Cosine Similarity
</div>
""", unsafe_allow_html=True)


## 🔑 Step 7 — Configure ngrok and Launch the App

1. Sign up free at **https://ngrok.com**
2. Copy your token from **https://dashboard.ngrok.com/get-started/your-authtoken**
3. Paste it below, then run the launch cell

In [ ]:
from pyngrok import ngrok

# 🔑 Paste your ngrok auth token between the quotes below
NGROK_TOKEN = "paste_your_ngrok_token_here"

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok authentication configured!")

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Stop any existing Streamlit process first
get_ipython().system("pkill -f streamlit")
time.sleep(1)

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.address=0.0.0.0",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(4)

ngrok.kill()
public_url = ngrok.connect(8501)

print("=" * 60)
print("🤖  ASKAI CHATBOT IS LIVE!")
print(f"👉  Open this link:  {public_url}")
print("=" * 60)
print("ℹ️   Keep this cell running — closing/interrupting it stops the app.")

## ⏹️ Step 8 — Stop the App

In [ ]:
ngrok.disconnect(public_url.public_url)
ngrok.kill()
streamlit_process.terminate()

print("🛑 App stopped and tunnel closed.")